In [1]:
import pandas as pd
import os
from glob import glob
import numpy as np
from pandas.api.types import CategoricalDtype

In [2]:
df_1 = pd.read_parquet('회원_전처리_Segment.parquet')  
df_2 = pd.read_parquet('신용_전처리_Segment.parquet') 
df_3 = pd.read_parquet('승인_전처리_Segment.parquet')
df_4 = pd.read_parquet('청구_전처리_Segment.parquet')
df_5 = pd.read_parquet('잔액_전처리_Segment.parquet')
df_6 = pd.read_parquet('채널_전처리_Segment.parquet')
df_7 = pd.read_parquet('마케팅_전처리_Segment.parquet')
df_8 = pd.read_parquet('성과_전처리_Segment.parquet')

In [3]:
df_merged = (
    df_1
    .merge(df_2,  on=['기준년월','ID','Segment'], how='left')
    .merge(df_3,  on=['기준년월','ID','Segment'], how='left')
    .merge(df_4,  on=['기준년월','ID','Segment'], how='left')
    .merge(df_5,  on=['기준년월','ID','Segment'], how='left')
    .merge(df_6,  on=['기준년월','ID','Segment'], how='left')
    .merge(df_7,  on=['기준년월','ID','Segment'], how='left')
    .merge(df_8, on=['기준년월','ID','Segment'], how='left'))
print(df_merged.shape)
df_merged

(2400000, 138)


,기준년월,ID,Segment,소지카드수_이용가능_신용,입회일자_신용,수신거부여부_TM,유효카드수_신용체크,이용가능카드수_신용체크,이용카드수_신용,이용금액_R3M_신용,...,월중평잔_일시불,평잔_일시불_3M,평잔_일시불_6M,인입일수_ARS_R6M,방문후경과월_앱_R6M,불만제기후경과월_R12M,컨택건수_이용유도_TM_R6M,컨택건수_이용유도_EM_R6M,잔액_신판ca최대한도소진율_r6m,변동률_RV일시불평잔
0,201807,TRAIN_000000,D,1,20130101,0,2,2,1,196,...,1503,1791,2440,8,6,12,3,57,0.849842,0.999998
1,201807,TRAIN_000001,E,1,20170801,0,1,1,1,13475,...,4447,3761,2677,0,6,12,2,2,0.851009,1.092698
2,201807,TRAIN_000002,C,1,20080401,0,2,2,1,23988,...,5540,6796,9118,1,0,12,2,12,0.938161,1.006124
3,201807,TRAIN_000003,D,2,20160501,0,3,3,1,3904,...,606,772,884,10,6,12,2,35,1.135424,0.999998
4,201807,TRAIN_000004,E,1,20180601,0,2,2,0,0,...,0,0,21,0,6,0,7,0,0.000000,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,201812,TRAIN_399995,E,1,20010701,0,2,2,0,7267,...,0,0,0,0,6,0,0,0,0.032439,0.999998
2399996,201812,TRAIN_399996,D,1,20170701,0,1,1,1,27636,...,5515,9424,12524,0,6,12,0,58,0.168081,0.999998
2399997,201812,TRAIN_399997,C,1,20090501,1,1,1,1,23187,...,3046,2998,3241,0,6,12,0,0,0.190393,0.999998
2399998,201812,TRAIN_399998,E,1,20130101,1,1,1,0,0,...,0,0,0,0,6,0,0,0,0.012677,0.999998


In [4]:
ex1 = df_merged
ex1 = ex1.fillna(-1)

In [5]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()
low_var = num_df.var()[ num_df.var() < 0.001 ].index.tolist()
low_var

[]

In [6]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)


삭제 대상 컬럼 (결측>20% 또는 동일값>80%): []


In [7]:
import pandas as pd
import numpy as np

num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)


high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 474


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,이용건수_신용_R12M,이용건수_신판_R12M,0.999881,0.480732,0.476833
1,이용건수_신용_R6M,이용건수_신판_R6M,0.999862,0.436528,0.432737
2,이용건수_신용_R3M,이용건수_신판_R3M,0.999836,0.433638,0.429748
3,이용건수_신용_B0M,이용건수_신판_B0M,0.999802,0.436677,0.432548
4,이용건수_신판_R3M,이용건수_일시불_R3M,0.999771,0.429748,0.426715
...,...,...,...,...,...
469,이용금액_일시불_R12M,정상청구원금_B5M,0.800286,0.595858,0.660738
470,_1순위카드이용금액,이용건수_오프라인_R3M,0.800252,0.573870,0.449336
471,이용금액_일시불_B0M,정상입금원금_B0M,0.800075,0.575032,0.549280
472,이용금액_일시불_B0M,이용건수_일시불_R12M,0.800072,0.575032,0.472362


In [8]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    # 결측값 건너뛰기
    if pd.isna(c1) or pd.isna(c2):
        continue

    # 둘 중 하나라도 Segment와의 상관계수가 0.3 이상이면 건너뜀
    if abs(c1) >= 0.3 or abs(c2) >= 0.3:
        continue

    # 둘 다 0.3 미만일 때: Segment와 더 상관관계가 낮은 쪽 제거
    if abs(c1) < abs(c2):
        to_drop.append(f1)
    else:
        to_drop.append(f2)

# 중복 제거
to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)


▶ 제거 대상 피처 수: 0
제거할 피처 목록:
[]


In [9]:
cols_to_drop = ['_1순위카드이용건수', '잔액_할부_B0M', '청구금액_R6M']
ex1.drop(columns=cols_to_drop, inplace=True)

In [13]:
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

num_df = (
    ex1
    .select_dtypes(include=[np.number])
    .drop(columns=['기준년월'], errors='ignore')
    .dropna()
)


X = add_constant(num_df)

vif_df = pd.DataFrame({
    'feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i)
            for i in range(X.shape[1])]
})

vif_df = vif_df[vif_df['feature'] != 'const'].reset_index(drop=True)

high_vif = vif_df[vif_df['VIF'] > 10]

print("VIF > 10인 컬럼들:", high_vif['feature'].tolist())
print(high_vif)


C:\Users\AHN\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


VIF > 10인 컬럼들: []
Empty DataFrame
Columns: [feature, VIF]
Index: []


In [11]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)

In [12]:
ex1.to_parquet('Segment_merge_ver_05.parquet', index=False)

In [13]:
ex1.columns.tolist()

['기준년월',
 'ID',
 'Segment',
 '소지카드수_이용가능_신용',
 '입회일자_신용',
 '수신거부여부_TM',
 '유효카드수_신용체크',
 '이용가능카드수_신용체크',
 '이용카드수_신용',
 '이용금액_R3M_신용',
 '_1순위카드이용금액',
 '_1순위카드이용건수',
 '_2순위카드이용건수',
 '최종유효년월_신용_이용가능',
 '이용가능여부_해외겸용_본인',
 '이용여부_3M_해외겸용_본인',
 '카드이용한도금액',
 'CA한도금액',
 'CA이자율_할인전',
 '카드이용한도금액_B1M',
 '카드이용한도금액_B2M',
 '최종이용일자_CA',
 '최종이용일자_할부',
 '이용건수_신용_B0M',
 '이용건수_신판_B0M',
 '이용건수_일시불_B0M',
 '이용금액_일시불_B0M',
 '이용건수_신용_R12M',
 '이용건수_신판_R12M',
 '이용건수_일시불_R12M',
 '이용금액_일시불_R12M',
 '이용금액_할부_R12M',
 '이용금액_할부_무이자_R12M',
 '이용금액_체크_R12M',
 '최대이용금액_일시불_R12M',
 '최대이용금액_할부_R12M',
 '최대이용금액_할부_무이자_R12M',
 '이용개월수_신용_R12M',
 '이용개월수_할부_무이자_R12M',
 '이용건수_신용_R6M',
 '이용건수_신판_R6M',
 '이용건수_일시불_R6M',
 '이용금액_일시불_R6M',
 '이용금액_할부_무이자_R6M',
 '이용건수_신용_R3M',
 '이용건수_신판_R3M',
 '이용건수_일시불_R3M',
 '이용금액_일시불_R3M',
 '이용가맹점수',
 '쇼핑_도소매_이용금액',
 '쇼핑_마트_이용금액',
 '쇼핑_슈퍼마켓_이용금액',
 '쇼핑_편의점_이용금액',
 '쇼핑_온라인_이용금액',
 '쇼핑_기타_이용금액',
 '교통_주유이용금액',
 '교통_정비이용금액',
 '교통_택시이용금액',
 '납부_기타이용금액',
 '_1순위업종',
 '_1순위업종_이용금액',
 '_2순위업종_이용금액',
 '_3순위업종',
 '_

In [13]:
ex1

,기준년월,ID,Segment,입회일자_신용,수신거부여부_TM,유효카드수_체크,이용가능카드수_신용체크,이용카드수_신용체크,이용금액_R3M_신용체크,_2순위카드이용금액,...,혜택수혜금액_R3M,월중평잔,평잔_일시불_6M,인입일수_ARS_R6M,방문후경과월_앱_R6M,불만제기후경과월_R12M,컨택건수_이용유도_TM_R6M,컨택건수_이용유도_EM_R6M,잔액_신판ca최대한도소진율_r6m,변동률_RV일시불평잔
0,201807,TRAIN_000000,D,20130101,0,1,2,1,196,0,...,3,17237,2440,8,6,12,3,57,0.849842,0.999998
1,201807,TRAIN_000001,E,20170801,0,0,1,1,13475,0,...,0,7967,2677,0,6,12,2,2,0.851009,1.092698
2,201807,TRAIN_000002,C,20080401,0,1,2,1,23988,0,...,121,59917,9118,1,0,12,2,12,0.938161,1.006124
3,201807,TRAIN_000003,D,20160501,0,1,3,1,3904,0,...,3,27854,884,10,6,12,2,35,1.135424,0.999998
4,201807,TRAIN_000004,E,20180601,0,1,2,1,1190,0,...,0,0,21,0,6,0,7,0,0.000000,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,201812,TRAIN_399995,E,20010701,0,1,2,1,10755,0,...,0,0,0,0,6,0,0,0,0.032439,0.999998
2399996,201812,TRAIN_399996,D,20170701,0,0,1,1,27636,0,...,164,29429,12524,0,6,12,0,58,0.168081,0.999998
2399997,201812,TRAIN_399997,C,20090501,1,0,1,1,23187,0,...,0,7383,3241,0,6,12,0,0,0.190393,0.999998
2399998,201812,TRAIN_399998,E,20130101,1,0,1,0,0,0,...,0,0,0,0,6,0,0,0,0.012677,0.999998
